# 1. 智能体的创建

```python
create_agent(
  model: str | BaseChatModel,
  tools: Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None = None,
  *,
  system_prompt: str | SystemMessage | None = None,
  middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),
  response_format: ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None = None,
  state_schema: type[AgentState[ResponseT]] | None = None,
  context_schema: type[ContextT] | None = None,
  checkpointer: Checkpointer | None = None,
  store: BaseStore | None = None,
  interrupt_before: list[str] | None = None,
  interrupt_after: list[str] | None = None,
  debug: bool = False,
  name: str | None = None,
  cache: BaseCache[Any] | None = None
) -> CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]
```

In [3]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model 

model = init_chat_model(                    
    model="ollama:qwen3.5:0.8b ",
    temperature=0.7,
    base_url="http://localhost:11434"       #可选：默认，是环境变量
)

agent = create_agent(
    model=model
)

print(agent)

In [16]:
agent = create_agent(model="ollama:deepseek-r1:1.5b")

#messages = [{"role": "user", "content": "今天天气怎么样？"}]
response = agent.invoke(
    {"messages": [{"role": "user", "content": "今天天气怎么样？"}]}
)
print(response) 

{'messages': [HumanMessage(content='今天天气怎么样？', additional_kwargs={}, response_metadata={}, id='288130b4-d50e-46f0-a34d-01a234d9dc63'), AIMessage(content='您好！我是2024年7月2日 Update过的 AI，目前无法查询实时信息。您可以前往气象网站或天气APP获取最新的天气情况。', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-04-08T02:48:02.326789Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3683248250, 'load_duration': 2773022542, 'prompt_eval_count': 7, 'prompt_eval_duration': 172593792, 'eval_count': 40, 'eval_duration': 712926670, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--019d6afd-4db1-77e0-831f-a1089a156a65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 40, 'total_tokens': 47})]}


In [18]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool       #装饰器，标记这是一个工具函数
def get_weather(location: str) -> str:
    """
    获取指定位置的天气信息
    参数:
    location (str): 位置名称，例如城市名
    返回:
    str: 天气信息的字符串描述"""
    pass
    #return f"The weather in {location} is sunny."
    return "天气是晴朗的。"

# 创建一个具有工具函数的智能体
agent = create_agent(
    model="ollama:qwen3.5:0.8b",
    tools=[get_weather]                     #将工具函数传递给智能体
)

#print(agent)

#调用智能体，询问天气
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "请告诉我北京的天气。"}
        ]
    }
)
print(response["messages"][-1].content)

北京的天气信息如下：

**今日天气状况：晴朗**

您可以根据这个天气情况来安排活动哦！


# 2.智能代理综合例子

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def get_time() -> str:
    """ 
    获取当前时间的工具函数
    返回:
    str: 当前时间的字符串描述
    """
    pass
    #return "The current time is 3:00 PM."
    return "2026-04-08/15:00:00"

@tool
def control_system(oper: str) -> str:
    """
    控制系统的工具函数,包含关机，清空桌面，鼠标控制，键盘控制
    参数:
    oper (str): 要执行的系统命令
    返回:
    str: 系统命令执行结果的字符串描述
    """
    print(f"操作：{oper}")
    #return f"Executed command: {command}"
    return f"已执行命令：{oper}"

@tool
def calculator(op: str,a: float,b: float) -> float:
    """
    执行基本的数学运算
    """
    print(f"{a} {op} {b}=?")
    return 99.99


agent = create_agent(
    model="ollama:qwen3.5:0.8b",
    tools=[get_time, control_system, calculator]
)

response1 = agent.invoke({
    "messages": [
        {"role": "user", "content": "请告诉我现在的时间。"}
        ]
    }
)

response2 = agent.invoke({
    "messages": [
        {"role": "user", "content": "请帮清理桌面"}
        ]
    }
)   
response3 = agent.invoke({
    "messages": [
        {"role": "user", "content": "请计算一下5加3等于 多少。"}
        ]
    } 
)

print(response1["messages"][-1].content)
print(response2["messages"][-1].content)
print(response3["messages"][-1].content)

操作：清空桌面
5.0 + 3.0=?


TypeError: exceptions must derive from BaseException